 # Artificial Intelligence Technology and Application

 ## Machine Learning Lab Guide - Student Version

 # 1 Emotion Recognition of Customer Evaluations in the Retail Industry

 ## 1.1 Introduction

 Emotion analysis classification based on natural language processing (NLP).

 ## 1.2 Procedure

 ### 1.2.1 Data Management & 1.2.2 Data Reading

 Since we lack the original retail review dataset, we generate a small replica dataset mimicking it.

In [1]:
import pandas as pd
import numpy as np
import re
import string

# Mimic retail reviews dataset
df = pd.DataFrame({
    'reviews.rating': [1.0, 5.0, 4.0, 2.0, 5.0, 3.0, 4.0, 1.0, np.nan, 5.0],
    'reviews.text': [
        "disappointed, battery died in one week",
        "great for beginner or experienced person bought",
        "inexpensive tablet for him to use and learn on",
        "charger broke very fast... nope",
        "I've had my fire hd two weeks now and I love",
        "okay, ordinary item, nothing surprised.",
        "huge screen, nice experience but slightly bent",
        "terrible customer service and bad product",
        "will not buy again", # nan rating
        "this amazon fire 8 inch tablet is the perfect size!"
    ],
    'reviews.title': ["Bad", "Great", "Good value", "Broke", "Love it", "Ok", "Nice", "Terrible", "Nah", "Perfect"],
    'reviews.username': ["user1", "user2", "user3", "user4", "user5", "user6", "user7", "user8", "user9", "user10"]
})

print("Shape of original dataset:", df.shape)

# View missing values
print("\nMissing values:\n", df.isnull().sum())

# Drop nan ratings
senti = df.dropna(subset=['reviews.rating']).copy()
check = df[df['reviews.rating'].isnull()].copy()
print("\nShape after dropping nan ratings:", senti.shape)

# Convert ratings to pos/neg
senti['senti'] = np.where(senti['reviews.rating'] >= 4, 'pos', 'neg')


Shape of original dataset: (10, 4)

Missing values:
 reviews.rating      1
reviews.text        0
reviews.title       0
reviews.username    0
dtype: int64

Shape after dropping nan ratings: (9, 4)


 ### 1.2.3 Data Processing

In [2]:
# Clean up function
def cleanup(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

senti['Summary_Clean'] = senti['reviews.text'].apply(cleanup)
check['Summary_Clean'] = check['reviews.text'].apply(cleanup)

print("\nCleaned text sample:")
print(senti[['Summary_Clean', 'senti']].head())

# Train test split
split = senti[['Summary_Clean', 'senti']].copy()

train = split.sample(frac=0.8, random_state=42)
test = split.drop(train.index)

print("\nSize of train:", train.shape)
print("Size of test:", test.shape)



Cleaned text sample:
                                     Summary_Clean senti
0            disappointed battery died in one week   neg
1  great for beginner or experienced person bought   pos
2   inexpensive tablet for him to use and learn on   pos
3                     charger broke very fast nope   neg
4      ive had my fire hd two weeks now and i love   pos

Size of train: (7, 2)
Size of test: (2, 2)


 ### 1.2.4 Model Training

In [3]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.naive_bayes import MultinomialNB, BernoulliNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

%matplotlib inline

# Convert words to TF matrix and IDF
vectorizer = CountVectorizer()
X_train_counts = vectorizer.fit_transform(train['Summary_Clean'])
X_test_counts = vectorizer.transform(test['Summary_Clean'])
X_check_counts = vectorizer.transform(check['Summary_Clean'])

tfidf_transformer = TfidfTransformer()
X_train_tfidf = tfidf_transformer.fit_transform(X_train_counts)
X_test_tfidf = tfidf_transformer.transform(X_test_counts)
check_tfidf = tfidf_transformer.transform(X_check_counts)

# MultinomialNB
mnb = MultinomialNB()
mnb.fit(X_train_tfidf, train['senti'])
print("\nMultinomial Accuracy:", accuracy_score(test['senti'], mnb.predict(X_test_tfidf)))

# BernoulliNB
bnb = BernoulliNB()
bnb.fit(X_train_tfidf, train['senti'])
print("Bernoulli Accuracy:", accuracy_score(test['senti'], bnb.predict(X_test_tfidf)))

# Logistic Regression
lr_nlp = LogisticRegression()
lr_nlp.fit(X_train_tfidf, train['senti'])
print("Logistic Regression Accuracy:", accuracy_score(test['senti'], lr_nlp.predict(X_test_tfidf)))

# Verify model
predictions = lr_nlp.predict_proba(check_tfidf)
print("\nPredicted probabilities for Check dataset:")
for pred in predictions:
    print(f"Sample estimated as pos? - negative prob {pred[0]:.6f}, positive prob {pred[1]:.6f}")

# Wordcloud visualization
try:
    from wordcloud import WordCloud, STOPWORDS
    text_data = ' '.join(train['Summary_Clean'])
    wordcloud = WordCloud(stopwords=STOPWORDS, background_color='white', width=800, height=400).generate(text_data)
    
    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis("off")
    plt.title("Word Cloud")
    plt.show()
except ImportError:
    print("\nWordCloud module is not installed. To see the wordcloud visualization, run: pip install wordcloud")



Multinomial Accuracy: 0.5
Bernoulli Accuracy: 0.5
Logistic Regression Accuracy: 0.5

Predicted probabilities for Check dataset:
Sample estimated as pos? - negative prob 0.433010, positive prob 0.566990

WordCloud module is not installed. To see the wordcloud visualization, run: pip install wordcloud
